<a href="https://colab.research.google.com/github/Nikhil4002-50-82/Automatic-Pedicle-Screw-Planning/blob/main/AutomaticPedicleScrewPlanning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [26]:
!ls /content/drive/MyDrive/dataset

case_0000.nii  spine_2-label.nii


# ***Phase 1- Geometry Based Planning***

***Algorithm***

In [27]:
import numpy as np
import nibabel as nib
from scipy.ndimage import distance_transform_edt, label as cc_label
from scipy.ndimage import map_coordinates
from sklearn.decomposition import PCA


# File paths
# ctPath="/content/drive/MyDrive/SpineData/case_002/case_0000.nii"
# segPath="/content/drive/MyDrive/SpineData/case_002/spine_2-label.nii"

ctPath="/content/drive/MyDrive/dataset/case_0000.nii"
segPath="/content/drive/MyDrive/dataset/spine_2-label.nii"

voxelThreshold=5000

# Available screw diameters
globalDiameters=[8.5,7.5,7.0,6.5,5.5,5.0,4.5,4.0]

# Maximum allowed diameter per vertebra
maxDiameterPerLevel={
"L1":6.5,
"L2":7.0,
"L3":7.5,
"L4":8.5,
"L5":8.5
}

stepMM=0.5
minLengthMM=18

directionConeDegLR=35
directionConeDegSI=20

directionSamplesLR=13
directionSamplesSI=9


# Scoring weights
wDT=5.0
wLen=1.5
wTilt=0.5


labelMap={5:"L1",4:"L2",3:"L3",2:"L4",1:"L5"}


# Load NIFTI file (CT or segmentation)
def loadNifti(path):

    nii=nib.load(path)

    return nii.get_fdata(), nii.header.get_zooms(), nii.affine


ct,spacingCT,affineCT=loadNifti(ctPath)
seg,spacing,affine=loadNifti(segPath)


# Keep only valid vertebrae with enough voxels
def getValidLabels(seg):

    valid=[]

    for labelVal in range(1,6):

        mask=(seg==labelVal)

        if np.sum(mask)==0:
            continue

        labeled,nc=cc_label(mask)

        sizes=np.bincount(labeled.ravel())
        sizes[0]=0

        largest=np.argmax(sizes)

        component=(labeled==largest)

        if np.sum(component)>voxelThreshold:

            valid.append((labelVal,component))

    return valid


validSegments=getValidLabels(seg)


# Compute local coordinate system of vertebra
# (Superior-Inferior, Left-Right, Anterior-Posterior)
def computeStableFrame(mask,affine):

    coords=np.argwhere(mask)

    coordsWorld=nib.affines.apply_affine(affine,coords)

    centroid=coordsWorld.mean(axis=0)

    pca=PCA(n_components=3)
    pca.fit(coordsWorld-centroid)

    axes=pca.components_

    worldZ=np.array([0,0,1])

    siAxis=axes[np.argmax(np.abs(axes@worldZ))]

    if np.dot(siAxis,worldZ)<0:
        siAxis=-siAxis


    tempAxis=axes[np.argmin(np.abs(axes@worldZ))]

    lrAxis=tempAxis - np.dot(tempAxis,siAxis)*siAxis
    lrAxis/=np.linalg.norm(lrAxis)

    apAxis=np.cross(siAxis,lrAxis)
    apAxis/=np.linalg.norm(apAxis)

    return centroid,np.vstack([siAxis,lrAxis,apAxis])


# Compute distance from every voxel to bone boundary
def computeDistance(mask):

    return distance_transform_edt(mask,sampling=spacing)



# Detect left and right pedicle centers
# Uses thickest bone region inside pedicle
def pedicleCenters(mask,dist,centroid,axes,affine):

    coords=np.argwhere(mask)

    coordsWorld=nib.affines.apply_affine(affine,coords)

    siAxis,lrAxis,apAxis=axes

    rel=coordsWorld-centroid

    siVals=rel@siAxis
    lrVals=rel@lrAxis
    apVals=rel@apAxis


    midMask=np.abs(siVals)<np.percentile(np.abs(siVals),35)
    posteriorMask=apVals<np.percentile(apVals,40)

    leftMask=lrVals<0
    rightMask=lrVals>0


    leftCoords=coords[midMask & posteriorMask & leftMask]
    rightCoords=coords[midMask & posteriorMask & rightMask]


    if len(leftCoords)<30 or len(rightCoords)<30:

        return None,None


    lVox=leftCoords[np.argmax(dist[leftCoords[:,0],
                                      leftCoords[:,1],
                                      leftCoords[:,2]])]

    rVox=rightCoords[np.argmax(dist[rightCoords[:,0],
                                       rightCoords[:,1],
                                       rightCoords[:,2]])]


    lMM=nib.affines.apply_affine(affine,lVox)
    rMM=nib.affines.apply_affine(affine,rVox)

    return lMM,rMM



# Find entry point on posterior surface
# Move backward from pedicle center until outside bone
def findEntry(center,axes,maskFloat,affine):

    siAxis,lrAxis,apAxis=axes

    direction=-apAxis

    invAff=np.linalg.inv(affine)

    p=center.copy()

    for _ in range(300):

        vox=nib.affines.apply_affine(invAff,p)

        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break

        val=map_coordinates(maskFloat,
                            [[vox[0]],[vox[1]],[vox[2]]],
                            order=1)[0]

        if val<0.5:
            break

        p+=direction*stepMM

    return p+apAxis*1.0



# Check if screw cylinder stays inside bone
def cylinderSafe(p,d,radius,maskFloat,affine):

    invAff=np.linalg.inv(affine)

    for angle in np.linspace(0,2*np.pi,8,endpoint=False):

        offset=radius*(np.cos(angle)*np.cross(d,[0,0,1])
                       +np.sin(angle)*np.cross(d,[1,0,0]))

        testPoint=p+offset

        vox=nib.affines.apply_affine(invAff,testPoint)

        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            return False

        val=map_coordinates(maskFloat,
                            [[vox[0]],[vox[1]],[vox[2]]],
                            order=1)[0]

        if val<0.5:
            return False

    return True



# Evaluate one screw trajectory
# Computes length and safety margin
def evaluate(entry,direction,maskFloat,dist,affine,radius,axes):

    d=direction/np.linalg.norm(direction)

    invAff=np.linalg.inv(affine)

    t=0
    minDT=999


    while True:

        # Ignore first 5 mm (entry cortex)
        if t<5:
            t+=stepMM
            continue


        p=entry+d*t

        vox=nib.affines.apply_affine(invAff,p)

        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break


        maskVal=map_coordinates(maskFloat,
                                 [[vox[0]],[vox[1]],[vox[2]]],
                                 order=1)[0]

        if maskVal<0.5:
            break


        dtVal=map_coordinates(dist,
                               [[vox[0]],[vox[1]],[vox[2]]],
                               order=1)[0]

        minDT=min(minDT,dtVal)


        if not cylinderSafe(p,d,radius,maskFloat,affine):
            break


        t+=stepMM


    if t<minLengthMM:
        return None


    # Reject unsafe screws
    if minDT-radius<=0:
        return None


    siAxis=axes[0]

    tilt=abs(np.dot(d,siAxis))


    score=wDT*minDT + wLen*(t/10) - wTilt*tilt


    return score,t,minDT,p,d



# Try many directions and diameters
# Select best safe screw
def optimize(center,axes,maskFloat,dist,affine,diameters):

    siAxis,lrAxis,apAxis=axes

    entry=findEntry(center,axes,maskFloat,affine)

    best=None


    for diam in diameters:

        radius=diam/2


        for lrAng in np.linspace(-directionConeDegLR,
                                  directionConeDegLR,
                                  directionSamplesLR):

            for siAng in np.linspace(-directionConeDegSI,
                                      directionConeDegSI,
                                      directionSamplesSI):

                direction=apAxis \
                    + np.tan(np.deg2rad(lrAng))*lrAxis \
                    + np.tan(np.deg2rad(siAng))*siAxis


                r=evaluate(entry,direction,
                           maskFloat,dist,
                           affine,radius,axes)

                if r is None:
                    continue


                score,length,minDT,tip,d=r


                if best is None or score>best[0]:

                    best=(score,entry,tip,length,minDT,diam)


    return best



# MAIN PROGRAM
print("\nFINAL ROBUST PEDICLE SCREW PLANNER:\n")


for labelVal,mask in validSegments:

    name=labelMap.get(labelVal,str(labelVal))

    maxDiam=maxDiameterPerLevel[name]

    diameters=[d for d in globalDiameters if d<=maxDiam]


    print("==============")
    print(name)
    print("==============")

    print("Tested Diameters:",diameters)


    centroid,axes=computeStableFrame(mask,affine)

    dist=computeDistance(mask)

    maskFloat=mask.astype(np.float32)


    lCenter,rCenter=pedicleCenters(mask,dist,
                                      centroid,axes,
                                      affine)


    if lCenter is None:

        print("Pedicle detection failed\n")

        continue


    for side,center in [("Left",lCenter),
                        ("Right",rCenter)]:


        result=optimize(center,
                        axes,
                        maskFloat,
                        dist,
                        affine,
                        diameters)


        if result is None:

            print(f"{side}: NO SAFE PATH\n")

            continue


        score,entry,tip,length,minDT,diam=result


        print(f"{side} Screw Found")

        print("Diameter:",diam,"mm")

        print("Length:",round(length,1),"mm")

        print("Safety Margin:",
              round(minDT-diam/2,2),"mm")

        print()


FINAL ROBUST PEDICLE SCREW PLANNER:

L5
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 7.5 mm
Length: 21.5 mm
Safety Margin: 0.15 mm

Right Screw Found
Diameter: 8.5 mm
Length: 45.5 mm
Safety Margin: 0.21 mm

L4
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 44.5 mm
Safety Margin: 0.52 mm

Right Screw Found
Diameter: 7.0 mm
Length: 44.5 mm
Safety Margin: 0.48 mm

L3
Tested Diameters: [7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 49.0 mm
Safety Margin: 0.18 mm

Right Screw Found
Diameter: 5.5 mm
Length: 56.0 mm
Safety Margin: 0.39 mm

L2
Tested Diameters: [7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 46.0 mm
Safety Margin: 0.4 mm

Right Screw Found
Diameter: 6.5 mm
Length: 41.0 mm
Safety Margin: 0.44 mm

L1
Tested Diameters: [6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 43.5 mm
Safety Margin: 0.36 mm

Right Screw F

***Algorithm that supports visualization - TotalSegmentator***

In [28]:
import numpy as np
import nibabel as nib
from scipy.ndimage import distance_transform_edt, label as cc_label
from scipy.ndimage import map_coordinates
from sklearn.decomposition import PCA


# File paths
# ctPath="/content/drive/MyDrive/SpineData/case_002/case_0000.nii"
# segPath="/content/drive/MyDrive/SpineData/case_002/spine_2-label.nii"

ctPath="/content/drive/MyDrive/dataset/case_0000.nii"
segPath="/content/drive/MyDrive/dataset/spine_2-label.nii"

voxelThreshold=5000


# Available screw diameters
globalDiameters=[8.5,7.5,7.0,6.5,5.5,5.0,4.5,4.0]


# Maximum allowed diameter per vertebra
maxDiameterPerLevel={
"L1":6.5,
"L2":7.0,
"L3":7.5,
"L4":8.5,
"L5":8.5
}


stepMM=0.5
minLengthMM=18


directionConeDegLR=35
directionConeDegSI=20


directionSamplesLR=13
directionSamplesSI=9


# Scoring weights
wDT=5.0
wLen=1.5
wTilt=0.5


labelMap={5:"L1",4:"L2",3:"L3",2:"L4",1:"L5"}

# Load NIFTI
def loadNifti(path):

    nii=nib.load(path)

    return nii.get_fdata(), nii.header.get_zooms(), nii.affine


ct,spacingCT,affineCT=loadNifti(ctPath)
seg,spacing,affine=loadNifti(segPath)


# Keep valid vertebrae
def getValidLabels(seg):

    valid=[]

    for labelVal in range(1,6):

        mask=(seg==labelVal)

        if np.sum(mask)==0:
            continue

        labeled,nc=cc_label(mask)

        sizes=np.bincount(labeled.ravel())
        sizes[0]=0

        largest=np.argmax(sizes)

        component=(labeled==largest)

        if np.sum(component)>voxelThreshold:

            valid.append((labelVal,component))

    return valid


validSegments=getValidLabels(seg)

# PCA coordinate system
def computeStableFrame(mask,affine):

    coords=np.argwhere(mask)

    coordsWorld=nib.affines.apply_affine(affine,coords)

    centroid=coordsWorld.mean(axis=0)

    pca=PCA(n_components=3)

    pca.fit(coordsWorld-centroid)

    axes=pca.components_

    worldZ=np.array([0,0,1])

    siAxis=axes[np.argmax(np.abs(axes@worldZ))]

    if np.dot(siAxis,worldZ)<0:
        siAxis=-siAxis


    tempAxis=axes[np.argmin(np.abs(axes@worldZ))]

    lrAxis=tempAxis - np.dot(tempAxis,siAxis)*siAxis
    lrAxis/=np.linalg.norm(lrAxis)

    apAxis=np.cross(siAxis,lrAxis)
    apAxis/=np.linalg.norm(apAxis)

    return centroid,np.vstack([siAxis,lrAxis,apAxis])


# Distance transform
def computeDistance(mask):

    return distance_transform_edt(mask,sampling=spacing)


# Pedicle centers
def pedicleCenters(mask,dist,centroid,axes,affine):

    coords=np.argwhere(mask)

    coordsWorld=nib.affines.apply_affine(affine,coords)

    siAxis,lrAxis,apAxis=axes

    rel=coordsWorld-centroid

    siVals=rel@siAxis
    lrVals=rel@lrAxis
    apVals=rel@apAxis


    midMask=np.abs(siVals)<np.percentile(np.abs(siVals),35)
    posteriorMask=apVals<np.percentile(apVals,40)

    leftMask=lrVals<0
    rightMask=lrVals>0


    leftCoords=coords[midMask & posteriorMask & leftMask]
    rightCoords=coords[midMask & posteriorMask & rightMask]


    if len(leftCoords)<30 or len(rightCoords)<30:

        return None,None


    lVox=leftCoords[np.argmax(dist[leftCoords[:,0],
                                      leftCoords[:,1],
                                      leftCoords[:,2]])]

    rVox=rightCoords[np.argmax(dist[rightCoords[:,0],
                                       rightCoords[:,1],
                                       rightCoords[:,2]])]


    lMM=nib.affines.apply_affine(affine,lVox)
    rMM=nib.affines.apply_affine(affine,rVox)

    return lMM,rMM


# Entry point
def findEntry(center,axes,maskFloat,affine):

    siAxis,lrAxis,apAxis=axes

    direction=-apAxis

    invAff=np.linalg.inv(affine)

    p=center.copy()

    for _ in range(300):

        vox=nib.affines.apply_affine(invAff,p)

        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break

        val=map_coordinates(maskFloat,
                            [[vox[0]],[vox[1]],[vox[2]]],
                            order=1)[0]

        if val<0.5:
            break

        p+=direction*stepMM

    return p+apAxis*1.0

# Cylinder safety
def cylinderSafe(p,d,radius,maskFloat,affine):

    invAff=np.linalg.inv(affine)

    for angle in np.linspace(0,2*np.pi,8,endpoint=False):

        offset=radius*(np.cos(angle)*np.cross(d,[0,0,1])
                       +np.sin(angle)*np.cross(d,[1,0,0]))

        testPoint=p+offset

        vox=nib.affines.apply_affine(invAff,testPoint)

        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            return False

        val=map_coordinates(maskFloat,
                            [[vox[0]],[vox[1]],[vox[2]]],
                            order=1)[0]

        if val<0.5:
            return False

    return True

# Evaluate screw
def evaluate(entry,direction,maskFloat,dist,affine,radius,axes):

    d=direction/np.linalg.norm(direction)

    invAff=np.linalg.inv(affine)

    t=0
    minDT=999


    while True:

        if t<5:
            t+=stepMM
            continue


        p=entry+d*t

        vox=nib.affines.apply_affine(invAff,p)

        if any(v<0 or v>=s-1 for v,s in zip(vox,maskFloat.shape)):
            break


        maskVal=map_coordinates(maskFloat,
                                 [[vox[0]],[vox[1]],[vox[2]]],
                                 order=1)[0]

        if maskVal<0.5:
            break


        dtVal=map_coordinates(dist,
                               [[vox[0]],[vox[1]],[vox[2]]],
                               order=1)[0]

        minDT=min(minDT,dtVal)


        if not cylinderSafe(p,d,radius,maskFloat,affine):
            break


        t+=stepMM


    if t<minLengthMM:
        return None


    if minDT-radius<=0:
        return None


    siAxis=axes[0]

    tilt=abs(np.dot(d,siAxis))


    score=wDT*minDT + wLen*(t/10) - wTilt*tilt


    return score,t,minDT,p,d


# Optimize screws
def optimize(center,axes,maskFloat,dist,affine,diameters):

    siAxis,lrAxis,apAxis=axes

    entry=findEntry(center,axes,maskFloat,affine)

    best=None


    for diam in diameters:

        radius=diam/2


        for lrAng in np.linspace(-directionConeDegLR,
                                  directionConeDegLR,
                                  directionSamplesLR):

            for siAng in np.linspace(-directionConeDegSI,
                                      directionConeDegSI,
                                      directionSamplesSI):

                direction=apAxis \
                    + np.tan(np.deg2rad(lrAng))*lrAxis \
                    + np.tan(np.deg2rad(siAng))*siAxis


                r=evaluate(entry,direction,
                           maskFloat,dist,
                           affine,radius,axes)

                if r is None:
                    continue


                score,length,minDT,tip,d=r


                if best is None or score>best[0]:

                    best=(score,entry,tip,length,minDT,diam)


    return best


# SAVE RESULTS
resultsList=[]


print("\nFINAL ROBUST PEDICLE SCREW PLANNER:\n")


for labelVal,mask in validSegments:

    name=labelMap.get(labelVal,str(labelVal))

    maxDiam=maxDiameterPerLevel[name]

    diameters=[d for d in globalDiameters if d<=maxDiam]


    print("==============")
    print(name)
    print("==============")

    print("Tested Diameters:",diameters)


    centroid,axes=computeStableFrame(mask,affine)

    dist=computeDistance(mask)

    maskFloat=mask.astype(np.float32)


    lCenter,rCenter=pedicleCenters(mask,dist,
                                      centroid,axes,
                                      affine)


    for side,center in [("Left",lCenter),
                        ("Right",rCenter)]:


        result=optimize(center,
                        axes,
                        maskFloat,
                        dist,
                        affine,
                        diameters)


        if result is None:

            print(f"{side}: NO SAFE PATH\n")

            continue


        score,entry,tip,length,minDT,diam=result


        resultsList.append({
            "vertebra":name,
            "side":side,
            "entry":entry,
            "tip":tip,
            "diameter":diam
        })


        print(f"{side} Screw Found")

        print("Diameter:",diam,"mm")

        print("Length:",round(length,1),"mm")

        print("Safety Margin:",
              round(minDT-diam/2,2),"mm")

        print()


FINAL ROBUST PEDICLE SCREW PLANNER:

L5
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 7.5 mm
Length: 21.5 mm
Safety Margin: 0.15 mm

Right Screw Found
Diameter: 8.5 mm
Length: 45.5 mm
Safety Margin: 0.21 mm

L4
Tested Diameters: [8.5, 7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 44.5 mm
Safety Margin: 0.52 mm

Right Screw Found
Diameter: 7.0 mm
Length: 44.5 mm
Safety Margin: 0.48 mm

L3
Tested Diameters: [7.5, 7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 49.0 mm
Safety Margin: 0.18 mm

Right Screw Found
Diameter: 5.5 mm
Length: 56.0 mm
Safety Margin: 0.39 mm

L2
Tested Diameters: [7.0, 6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 46.0 mm
Safety Margin: 0.4 mm

Right Screw Found
Diameter: 6.5 mm
Length: 41.0 mm
Safety Margin: 0.44 mm

L1
Tested Diameters: [6.5, 5.5, 5.0, 4.5, 4.0]
Left Screw Found
Diameter: 6.5 mm
Length: 43.5 mm
Safety Margin: 0.36 mm

Right Screw F

In [29]:
import plotly.graph_objects as go
import numpy as np
import nibabel as nib

print("Creating Surgical-Grade Screw Visualization...")

# Function to generate screw cylinder
def createCylinder(entry, tip, diameter, resolution=20):

    entry=np.array(entry)
    tip=np.array(tip)

    direction=tip-entry
    length=np.linalg.norm(direction)

    direction=direction/length


    # Create perpendicular vectors
    if abs(direction[2])<0.9:
        v=np.cross(direction,[0,0,1])
    else:
        v=np.cross(direction,[0,1,0])

    v=v/np.linalg.norm(v)

    w=np.cross(direction,v)


    radius=diameter/2


    theta=np.linspace(0,2*np.pi,resolution)
    z=np.linspace(0,length,2)


    theta,z=np.meshgrid(theta,z)


    x=radius*np.cos(theta)
    y=radius*np.sin(theta)


    X=entry[0]+direction[0]*z + v[0]*x + w[0]*y
    Y=entry[1]+direction[1]*z + v[1]*x + w[1]*y
    Z=entry[2]+direction[2]*z + v[2]*x + w[2]*y


    return X,Y,Z


# Spine surface
coords=np.argwhere(seg>0)

coords=coords[::10]

coordsWorld=nib.affines.apply_affine(affine,coords)


fig=go.Figure()


fig.add_trace(go.Scatter3d(

x=coordsWorld[:,0],
y=coordsWorld[:,1],
z=coordsWorld[:,2],

mode='markers',

marker=dict(
size=2,
color='lightgray',
opacity=0.15
),

name='Spine'

))

# Plot Cylindrical Screws
for r in resultsList:

    entry=r["entry"]
    tip=r["tip"]
    diam=r["diameter"]


    X,Y,Z=createCylinder(entry,tip,diam)


    fig.add_trace(go.Surface(

        x=X,
        y=Y,
        z=Z,

        opacity=1,

        showscale=False

    ))

# Layout
fig.update_layout(

title="3D Pedicle Screw Surgical Visualization",

scene=dict(
aspectmode='data'
),

height=900

)


fig.show()

Creating Surgical-Grade Screw Visualization...


In [ ]:
!pip install TotalSegmentator
!pip install scikit-image

import os

print("Activating TotalSegmentator Student License...")

os.environ["TOTALSEG_LICENSE"] = "aca_V4E9OLA5VBGIQO"

print("License Activated")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.9/212.9 kB 4.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Activating TotalSegmentator Student License...
License Activated


In [30]:
import os

print("Running TotalSegmentator Lumbar Vertebra Segmentation...")


inputCT="/content/drive/MyDrive/SpineData/case_002/case_0000.nii"

outputFolder="/content/totalseg_out"


!rm -rf "$outputFolder"
!mkdir -p "$outputFolder"


!TotalSegmentator \
-i "$inputCT" \
-o "$outputFolder" \
-ta total \
-rs vertebrae_L1 vertebrae_L2 vertebrae_L3 vertebrae_L4 vertebrae_L5


print("Segmentation Completed")

Running TotalSegmentator Lumbar Vertebra Segmentation...
No GPU detected. Running on CPU. This can be very slow. The '--fast' or the `--roi_subset` option can help to reduce runtime.

If you use this tool please cite: https://pubs.rsna.org/doi/10.1148/ryai.230024

Generating rough segmentation for cropping...
Resampling...
  Resampled in 3.67s
Predicting...
100% 12/12 [00:34<00:00,  2.86s/it]
  Predicted in 51.71s
Resampling...
  cropping from (366, 424, 283) to (82, 99, 139)
Resampling...
  Resampled in 0.00s
Predicting part 1 of 1 ...
100% 2/2 [00:45<00:00, 22.70s/it]
  Predicted in 70.58s
Resampling...
Saving segmentations...
  Saved in 9.34s
Segmentation Completed


In [31]:
import nibabel as nib
import numpy as np
from skimage.measure import marching_cubes

print("Building Smooth Vertebra Surface Mesh...")

totalsegFolder="/content/totalseg_out"

# Load Vertebrae L1-L5
L1=nib.load(totalsegFolder+"/vertebrae_L1.nii.gz").get_fdata()
L2=nib.load(totalsegFolder+"/vertebrae_L2.nii.gz").get_fdata()
L3=nib.load(totalsegFolder+"/vertebrae_L3.nii.gz").get_fdata()
L4=nib.load(totalsegFolder+"/vertebrae_L4.nii.gz").get_fdata()
L5=nib.load(totalsegFolder+"/vertebrae_L5.nii.gz").get_fdata()


# Combine Vertebrae
vertebraMask=L1+L2+L3+L4+L5

# Create Smooth Surface Mesh
verts,faces,_,_=marching_cubes(vertebraMask,level=0.5)

# Convert to World Coordinates
vertsWorld=nib.affines.apply_affine(affine,verts)

print("Mesh Ready")

Building Smooth Vertebra Surface Mesh...
Mesh Ready


In [32]:
import plotly.graph_objects as go
import numpy as np

print("Creating Surgical-Grade Visualization...")


# Screw Cylinder Function
def createCylinder(entry,tip,diameter,resolution=40):

    entry=np.array(entry)
    tip=np.array(tip)

    direction=tip-entry
    length=np.linalg.norm(direction)

    direction=direction/length


    if abs(direction[2])<0.9:
        v=np.cross(direction,[0,0,1])
    else:
        v=np.cross(direction,[0,1,0])

    v=v/np.linalg.norm(v)

    w=np.cross(direction,v)


    radius=diameter/2


    theta=np.linspace(0,2*np.pi,resolution)
    z=np.linspace(0,length,30)

    theta,z=np.meshgrid(theta,z)


    x=radius*np.cos(theta)
    y=radius*np.sin(theta)


    X=entry[0]+direction[0]*z + v[0]*x + w[0]*y
    Y=entry[1]+direction[1]*z + v[1]*x + w[1]*y
    Z=entry[2]+direction[2]*z + v[2]*x + w[2]*y


    return X,Y,Z

# Create Figure
fig=go.Figure()


# Smooth Vertebra Surface (TotalSegmentator)
fig.add_trace(go.Mesh3d(

x=vertsWorld[:,0],
y=vertsWorld[:,1],
z=vertsWorld[:,2],

i=faces[:,0],
j=faces[:,1],
k=faces[:,2],

opacity=0.25,
color='lightgray',

name="Lumbar Vertebrae"

))


# Cylindrical Screws
for r in resultsList:

    X,Y,Z=createCylinder(
        r["entry"],
        r["tip"],
        r["diameter"]
    )

    fig.add_trace(go.Surface(

        x=X,
        y=Y,
        z=Z,

        showscale=False,
        opacity=1

    ))


# Entry Points
for r in resultsList:

    entry=r["entry"]

    fig.add_trace(go.Scatter3d(

        x=[entry[0]],
        y=[entry[1]],
        z=[entry[2]],

        mode='markers',

        marker=dict(
            size=5,
            color='green'
        ),

        showlegend=False

    ))


# Layout
fig.update_layout(

title="TotalSegmentator + Pedicle Screw Planner",

scene=dict(
aspectmode='data'
),

height=900

)

fig.show()

Creating Surgical-Grade Visualization...
